# SQL_Data_Base

# 5. Connect_the_SQL_Data_Base

## Store and validate (2 marks). 

1. Write the three clean tables to data/clean/ as CSV files and to the SQLite database data/instamart.db as tables named customers, products, orders (for example with sqlite3.connect and DataFrame.to_sql, or an equivalent method; store order_ts as text in the form YYYY-MM-DD HH:MM:SS). 

2. Write a validation script that asserts at least eight of the reference checkpoints from tasks 3 and 4 and prints exactly ALL CHECKPOINTS PASSED when all hold.

In [1]:
import pandas as pd

In [24]:
from pathlib import Path
import pandas as pd
import sqlite3

DATA_DIR = Path("data/clean")

customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")

print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)

Customers: (1200, 5)
Products: (40, 4)
Orders: (12309, 10)


## 1. Create_Data_Base

In [3]:
from pathlib import Path
import sqlite3

# Create database folder
DB_DIR = Path("data")
DB_DIR.mkdir(parents=True, exist_ok=True)

# Database path
DB_PATH = DB_DIR / "instamart.db"

# Connect to SQLite database
conn = sqlite3.connect(DB_PATH)

# Create three tables
customers.to_sql("customers", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)

print("SQLite database created successfully!")
print("Database location:", DB_PATH)

SQLite database created successfully!
Database location: data\instamart.db


In [4]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

display(tables)

,name
0,customers
1,products
2,orders


In [5]:
db_path = "data/instamart.db" # create a path.

conn = sqlite3.connect(db_path) # connect a path to the database.

print("Data base Created and connected successfully.")

Data base Created and connected successfully.


## 2. Connect_to_the_database(product,customer,order_db)

In [7]:
customers.to_sql(
                        "customers",
                        conn,
                        if_exists="replace",
                        index=False
)

products.to_sql(
                        "products",
                        conn,
                        if_exists="replace",
                        index=False
)

orders.to_sql(
                        "orders",
                        conn,
                        if_exists="replace",
                        index=False
)

print("All Tables are stored Successfully.")

All Tables are stored Successfully.


In [8]:
tables = pd.read_sql_query(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = "table";
            """,
            conn
            
        )

display(tables)
print(tables)

,name
0,customers
1,products
2,orders


        name
0  customers
1   products
2     orders


In [9]:
print("Customes_Rows : " , pd.read_sql_query(
                    "SELECT COUNT(*) AS total FROM customers",conn
                    ).iloc[0,0])

print("products_Rows : " , pd.read_sql_query(
                    "SELECT COUNT(*) AS total FROM products",conn
                    ).iloc[0,0])

print("orders_Rows : " , pd.read_sql_query(
                    "SELECT COUNT(*) AS total FROM orders",conn
                    ).iloc[0,0])

Customes_Rows :  1200
products_Rows :  40
orders_Rows :  12309


In [10]:
print("Database exists:", DB_PATH.exists())
print("Database size:", DB_PATH.stat().st_size, "bytes")

Database exists: True
Database size: 991232 bytes


In [ ]:
pd.read_sql_query(
                    "SELECT * FROM orders LIMIT 5 ;",
                    conn
)

,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
0,ORD-110093,C0967,P020,1,2025-04-01 07:07:28,pending,15,127.50,NaN
1,ORD-110093,C0967,P020,1,2025-04-01 07:08:28,completed,15,127.50,19.0
2,ORD-111396,C1117,P036,2,2025-04-01 07:11:59,completed,0,1598.00,25.0
3,ORD-102006,C0194,P009,1,2025-04-01 07:12:43,cancelled,25,21.75,NaN
4,ORD-105921,C0569,P005,2,2025-04-01 07:54:43,completed,10,86.40,33.0


In [ ]:
pd.read_sql_query(
                "SELECT order_ts FROM orders LIMIT 5 ; ",
                conn
)

,order_ts
0,2025-04-01 07:07:28
1,2025-04-01 07:08:28
2,2025-04-01 07:11:59
3,2025-04-01 07:12:43
4,2025-04-01 07:54:43


## 6. SQL analysis (5 marks). Save seven queries in sql/queries.sql, run each from Python with pd.read_sql_query against data/instamart.db, and report the stated result for each. Unless stated otherwise, only completed orders count as revenue or spend.

Q1 — WHERE then HAVING. Keep only completed orders with order_amount > 500 (WHERE), group by customer, and keep customers whose total of those orders exceeds 8000 (HAVING). Reference: 30 customers.

Q2 — three-table join. Revenue by city and product category (join orders, customers, products). Reference: 30 rows; the 30 revenues add up to INR 2,998,668.80; the largest cell is Bengaluru / Electronics at INR 233,909.45.

Q3 — scalar subquery. Products priced above the average catalogue price. Reference: 11 products.

Q4 — LEFT JOIN. Customers who have no completed order. Reference: 25 customers.

Q5 — common table expression. One CTE computes each customer's total completed spend and a second computes the average of those totals; the main query 
returns customers whose total is above the average. Reference: 550 customers; average INR 2,552.06 (rounded to 2 decimals).

Q6 — ranking window functions. For each customer with completed spend, compute total spend and then RANK(), DENSE_RANK() and ROW_NUMBER() using OVER (PARTITION BY city ORDER BY total_spend DESC). Reference: the top customer per city is Bengaluru C0082 (12,856.20), Chennai C0930 (10,151.40), Delhi C0291 (13,742.60), Hyderabad C0743 (12,730.05), Mumbai C0693 (14,766.55), Pune C0426 (10,377.10); exactly 8 rows have RANK() different from ROW_NUMBER(), and your README states in one sentence why ties cause that.

Q7 — running total. Daily completed revenue (the day is the first ten characters of order_ts) with a cumulative revenue column computed as SUM(revenue) OVER (ORDER BY day). Reference: 183 daily rows; the last cumulative value is INR 2,998,668.80.

In [11]:
# Q_1 _ WHERE then HAVING

query_1 = pd.read_sql_query(
    """
    SELECT
        customer_id,
        ROUND(SUM(order_amount), 2) AS total_spend
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
    HAVING SUM(order_amount) > 8000
    ORDER BY total_spend DESC;
    """,
    conn
)

display(query_1)
# Q2 — three-table join.

query_2 = pd.read_sql_query(
                            """
                            SELECT c.city, p.category, ROUND(SUM(o.order_amount),2) AS revenue
                            FROM orders o JOIN customers c USING(customer_id) JOIN products p USING(product_id)
                            WHERE o.status='completed'
                            GROUP BY c.city,p.category ORDER BY c.city,p.category;
                            """,
                            conn
            )

display(query_2)

# Q3: products above average catalogue price

query_3 = pd.read_sql_query(
                            """
                            SELECT product_id, product_name, price_inr FROM products
                            WHERE price_inr > (SELECT AVG(price_inr) FROM products);
                            """,
                            conn
            )

display(query_3)

# Q4: customers without completed orders

query_4 = pd.read_sql_query(
                        """
                        SELECT c.customer_id FROM customers c LEFT JOIN orders o
                        ON c.customer_id=o.customer_id AND o.status='completed'
                        WHERE o.order_id IS NULL;
                        """,
                        conn
        )

display(query_4)

# -- Q5: CTE for above-average customer spend


query_5 = pd.read_sql_query(
                            """
                            WITH customer_spend AS (
                             SELECT customer_id, SUM(order_amount) AS total_spend FROM orders
                             WHERE status='completed' GROUP BY customer_id
                            ), average_spend AS (SELECT AVG(total_spend) AS avg_spend FROM customer_spend)
                            SELECT customer_id,total_spend FROM customer_spend,average_spend
                            WHERE total_spend > avg_spend;
                            """,
                            conn
        )

display(query_5)

# -- Q6: ranking window functions within city

query_6 = pd.read_sql_query(
                            """
                            WITH spend AS (
                             SELECT c.customer_id,c.city,SUM(o.order_amount) AS total_spend
                             FROM customers c JOIN orders o USING(customer_id)
                             WHERE o.status='completed' GROUP BY c.customer_id,c.city
                            )
                            SELECT *, RANK() OVER(PARTITION BY city ORDER BY total_spend DESC) AS spend_rank,
                             DENSE_RANK() OVER(PARTITION BY city ORDER BY total_spend DESC) AS dense_rank,
                             ROW_NUMBER() OVER(PARTITION BY city ORDER BY total_spend DESC) AS row_num
                            FROM spend;
                            """,
                            conn
        )

display(query_6)

# -- Q7: daily running total

query_7 = pd.read_sql_query(
                            """
                            WITH daily AS (
                             SELECT substr(order_ts,1,10) AS day,SUM(order_amount) AS revenue
                             FROM orders WHERE status='completed' GROUP BY substr(order_ts,1,10)
                            )
                            SELECT day,revenue,SUM(revenue) OVER(ORDER BY day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_revenue
                            FROM daily ORDER BY day;
                            """,
                            conn
        )

display(query_7)

,customer_id,total_spend
0,C0693,14766.55
1,C0291,13742.60
2,C0488,13187.50
3,C0082,12856.20
4,C0743,12730.05
5,C0525,12719.70
6,C0699,12471.80
7,C0267,12210.25
8,C0332,11806.40
9,C0512,11617.65


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


,product_id,product_name,price_inr
0,P014,Ghee 500ml,340
1,P030,Body Lotion 400ml,350
2,P032,Sunscreen 50g,399
3,P033,Wireless Earbuds,1499
4,P034,Power Bank 10000mAh,1299
5,P035,USB-C Cable 1m,299
6,P036,Phone Charger 20W,799
7,P037,Smart Bulb,599
8,P038,Bluetooth Speaker,1999
9,P039,Phone Stand,349


,customer_id
0,C0029
1,C0037
2,C0093
3,C0115
4,C0120
5,C0151
6,C0354
7,C0363
8,C0412
9,C0461


,customer_id,total_spend
0,C0002,4644.90
1,C0003,3644.85
2,C0007,2600.00
3,C0008,3096.50
4,C0009,8424.80
...,...,...
545,C1191,4570.00
546,C1197,3397.15
547,C1198,2803.20
548,C1199,3861.85


,customer_id,city,total_spend,spend_rank,dense_rank,row_num
0,C0082,Bengaluru,12856.20,1,1,1
1,C0525,Bengaluru,12719.70,2,2,2
2,C0699,Bengaluru,12471.80,3,3,3
3,C0543,Bengaluru,11195.10,4,4,4
4,C1147,Bengaluru,10156.90,5,5,5
...,...,...,...,...,...,...
1170,C0735,Pune,39.25,217,213,217
1171,C0541,Pune,38.25,218,214,218
1172,C1061,Pune,34.80,219,215,219
1173,C1158,Pune,33.60,220,216,220


,day,revenue,cumulative_revenue
0,2025-04-01,16876.50,16876.50
1,2025-04-02,19854.55,36731.05
2,2025-04-03,28048.35,64779.40
3,2025-04-04,14249.60,79029.00
4,2025-04-05,16031.50,95060.50
...,...,...,...
178,2025-09-26,17190.85,2942763.75
179,2025-09-27,13689.95,2956453.70
180,2025-09-28,12689.85,2969143.55
181,2025-09-29,14022.15,2983165.70


In [12]:
from pathlib import Path

clean_dir = Path("data/clean")

for file in clean_dir.iterdir():
    if file.is_file():
        print(file.name)

customers.csv
Customer_data_Cleaned.csv
orders.csv
Order_details_cleaned.csv
products.csv
Product_Cleaned.csv


In [13]:
from pathlib import Path

clean_dir = Path("data/clean")
clean_dir.mkdir(parents=True, exist_ok=True)

customers.to_csv(clean_dir / "customers.csv", index=False)
products.to_csv(clean_dir / "products.csv", index=False)
orders.to_csv(clean_dir / "orders.csv", index=False)

print("All three cleaned CSV files saved successfully!")

All three cleaned CSV files saved successfully!


In [14]:
for filename in ["customers.csv", "products.csv", "orders.csv"]:
    print(filename, ":", (clean_dir / filename).exists())

customers.csv : True
products.csv : True
orders.csv : True


# 7. pandas cross-check and exploratory analysis (2 marks). 

1. (a) Rebuild Q2 with pd.pivot_table(..., margins=True) and show that its grand total equals the SQL total; with groupby and named aggregation report, per city, the sum, mean and count of completed order_amount (reference: Bengaluru sum 523,643.40, mean 276.47, count 1,894; Mumbai sum 479,058.50, mean 297.00, count 1,613).
2. (b) For completed order_amount, report mean, median, skewness, the quartiles, the 1.5×IQR fences and the number of outliers beyond each fence, then answer in your own words, in at most 200 words in total,
3. the four questions:


  3.1. What is this data about?

  3.2. What do its distributions and statistics look like?
  
  3.3. Are there outliers or anomalies? What relationships exist between variables?
  
  3.4. For the last question use the correlation matrix of order_amount, discount_pct, quantity, delivery_minutes over completed orders and name the pair with the largest correlation in absolute value. Reference: mean 269.91 versus median 135.00 (right-skewed), skewness about 4.61, Q1 64.80, Q3 297.00, upper fence 645.30 with 897 outliers above it and none below; the correlation largest in absolute value is order_amount with discount_pct at about −0.095 (the pair order_amount and quantity is +0.087, slightly smaller in absolute value), and your answer must say that correlation of this size does not establish causation.

### Total revenue depand upon the catogery and city wise

In [15]:
revenue_city_category = (
    orders[orders["status"] == "completed"]
    .merge(customers[["customer_id", "city"]], on="customer_id")
    .merge(products[["product_id", "category"]], on="product_id")
    .groupby(["city", "category"])["order_amount"]
    .sum()
    .round(2)
    .reset_index(name="total_revenue")
    .sort_values(["city", "total_revenue"], ascending=[True, False])
)

display(revenue_city_category)

,city,category,total_revenue
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
0,Bengaluru,Dairy,87075.05
4,Bengaluru,Snacks,71549.40
3,Bengaluru,Personal Care,20193.85
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
5,Chennai,Dairy,78301.80
9,Chennai,Snacks,68325.70
8,Chennai,Personal Care,27675.40


### Customer_spending_by_city

In [16]:
customer_spending_by_city = (
    orders[orders["status"] == "completed"]
    .groupby("customer_id")["order_amount"]
    .sum()
    .reset_index(name="total_spend")
    .merge(
        customers[["customer_id", "city"]],
        on="customer_id"
    )
    .groupby("city")["total_spend"]
    .sum()
    .round(2)
    .reset_index()
    .sort_values("total_spend", ascending=False)
)

display(customer_spending_by_city)

,city,total_spend
3,Hyderabad,528589.90
0,Bengaluru,523643.40
2,Delhi,502829.80
5,Pune,500918.05
4,Mumbai,479058.50
1,Chennai,463629.15


In [18]:
completed_orders = orders[orders["status"] == "completed"]

In [19]:
amount_stats = completed_orders["order_amount"].describe()

display(amount_stats)

count    11110.000000
mean       269.907183
std        432.069126
min         12.000000
25%         64.800000
50%        135.000000
75%        297.000000
max       3998.000000
Name: order_amount, dtype: float64

In [20]:
Q1 = completed_orders["order_amount"].quantile(0.25)
Q3 = completed_orders["order_amount"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = completed_orders[
    (completed_orders["order_amount"] < lower_limit) |
    (completed_orders["order_amount"] > upper_limit)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)
print("Total outliers:", len(outliers))

Q1: 64.8
Q3: 297.0
IQR: 232.2
Lower limit: -283.49999999999994
Upper limit: 645.3
Total outliers: 897


In [21]:
correlation_data = completed_orders[
    ["order_amount", "quantity", "discount_pct", "delivery_minutes"]
].corr(numeric_only=True)

display(correlation_data.round(3))

,order_amount,quantity,discount_pct,delivery_minutes
order_amount,1.000,0.087,-0.095,-0.003
quantity,0.087,1.000,-0.000,-0.013
discount_pct,-0.095,-0.000,1.000,-0.010
delivery_minutes,-0.003,-0.013,-0.010,1.000


## Extract correlations with order amount

In [22]:
order_amount_corr = correlation_data["order_amount"].drop("order_amount")

display(order_amount_corr.sort_values(ascending=False))

quantity            0.087294
delivery_minutes   -0.003135
discount_pct       -0.095106
Name: order_amount, dtype: float64

## Identify the strongest relationship

In [23]:
strongest_corr = order_amount_corr.abs().idxmax()

print("Strongest correlation variable:", strongest_corr)

print(
    "Correlation value:",
    round(correlation_data.loc["order_amount", strongest_corr], 3)
)

Strongest correlation variable: discount_pct
Correlation value: -0.095


# Discriptive

### Exploratory Data Analysis and Interpretation

1. **What is this data about?**

   The dataset represents Instamart's completed customer orders. It contains information about order amounts, quantities, discounts and delivery times. It helps understand customer purchasing behaviour and business revenue.

2. **What do its distributions and statistics look like?**

   The average order amount is INR 269.91, while the median is INR 135.00. The skewness is approximately 4.61, indicating a positively (right) skewed distribution. The first quartile (Q1) is INR 64.80 and the third quartile (Q3) is INR 297.00.

3. **Are there outliers or anomalies?**

   Using the 1.5 × IQR method, the upper fence is INR 645.30. There are 897 outliers above the upper fence and no outliers below the lower fence. This indicates that some orders have unusually high amounts.

4. **What relationships exist between variables?**

   The correlation matrix shows that order amount and discount percentage have the strongest correlation in absolute value, approximately -0.095. Order amount and quantity have a correlation of +0.087. Both relationships are very weak. Correlation of this size does not establish causation.

# ---------- PART A IS DONE ----------